## 01. Create Discovery Golden Dataset

### Why are we implementing this?

We need a separate golden dataset to evaluate whether SEVA_AI can identify relevant government schemes from a citizen's real-world problem.

Unlike our existing FAQ dataset, citizen queries must not contain scheme names.

Each query will have manually verified relevance judgments for multiple schemes.

We will use this dataset to evaluate:
- Scheme discovery accuracy
- Recall@K and Precision@K
- MRR and MAP
- nDCG with graded relevance

We will initially target 150 citizen queries.

In [8]:
import json
from pathlib import Path

PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

DISCOVERY_DATASET_DIR = (
    PROJECT_ROOT
    / "notebooks"
    / "golden_dataset"
    / "discovery"
)

DISCOVERY_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DATASET_PATH = (
    DISCOVERY_DATASET_DIR
    / "discovery_golden_dataset.jsonl"
)

SCHEMA_PATH = (
    DISCOVERY_DATASET_DIR
    / "discovery_schema.json"
)

schema = {
    "query_id": "Unique query identifier",
    "citizen_query": "Citizen's actual problem without scheme names",
    "category": "Government scheme category",
    "answerable": "Boolean",
    "relevant_scheme_ids": [
        "Verified relevant scheme IDs"
    ],
    "relevance_judgments": [
        {
            "scheme_id": "Government scheme ID",
            "relevance_grade": 2,
            "rationale": "Why this scheme is relevant",
            "source_url": "Official verification URL"
        }
    ]
}

with open(SCHEMA_PATH, "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4, ensure_ascii=False)

if not DATASET_PATH.exists():
    DATASET_PATH.touch()

print("Discovery benchmark initialized!")
print("Dataset:", DATASET_PATH)
print("Schema:", SCHEMA_PATH)

Discovery benchmark initialized!
Dataset: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_golden_dataset.jsonl
Schema: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_schema.json


## 02. Analyze Scheme Categories

### Why are we implementing this?

We need to understand the distribution of government schemes before creating our 150 citizen queries.

We will inspect the existing scheme dataset to identify:
- Available scheme categories
- Number of schemes per category
- Available metadata fields
- Sample scheme records

This analysis will help us create a balanced discovery benchmark across different government domains.

## Full Scheme Dataset Validation

Validate the complete government scheme dataset before building the citizen-discovery benchmark. This checks data integrity, category coverage, scheme status, and geographic distribution.

In [10]:
import json
from pathlib import Path
from collections import Counter

SCHEME_PATH = Path(r"G:\SEVA_Ai\SEVA_Ai\notebooks\data\analysis\all_schemes.json")

with open(SCHEME_PATH, "r", encoding="utf-8") as file:
    schemes = json.load(file)

print("Total schemes:", len(schemes))
print("Unique scheme IDs:", len({s.get("scheme_id") for s in schemes}))
print("Categories:", len({
    category
    for scheme in schemes
    for category in scheme.get("categories", [])
}))

category_counts = Counter(
    category
    for scheme in schemes
    for category in scheme.get("categories", [])
)

status_counts = Counter(
    scheme.get("status", "unknown")
    for scheme in schemes
)

level_counts = Counter(
    scheme.get("level", "unknown")
    for scheme in schemes
)

state_counts = Counter(
    state
    for scheme in schemes
    for state in scheme.get("states", [])
)

print("\nCategory distribution:")
for category, count in category_counts.most_common():
    print(f"{category}: {count}")

print("\nStatus distribution:", status_counts)
print("\nLevel distribution:", level_counts)
print("\nTop 15 states:", state_counts.most_common(15))

Total schemes: 4428
Unique scheme IDs: 4428
Categories: 15

Category distribution:
Social welfare & Empowerment: 1331
Education & Learning: 1023
Agriculture,Rural & Environment: 808
Business & Entrepreneurship: 730
Women and Child: 426
Skills & Employment: 368
Banking,Financial Services and Insurance: 308
Health & Wellness: 263
Sports & Culture: 261
Housing & Shelter: 126
Science, IT & Communications: 114
Transport & Infrastructure: 101
Travel & Tourism: 95
Utility & Sanitation: 56
Public Safety,Law & Justice: 31

Status distribution: Counter({'active': 4384, 'closed': 44})

Level distribution: Counter({'State': 3735, 'Central': 693})

Top 15 states: [('All', 673), ('Gujarat', 594), ('Uttarakhand', 416), ('Madhya Pradesh', 263), ('Goa', 247), ('Puducherry', 245), ('Haryana', 224), ('Tamil Nadu', 212), ('Rajasthan', 147), ('West Bengal', 105), ('Chhattisgarh', 103), ('Bihar', 102), ('Assam', 86), ('Jharkhand', 86), ('Meghalaya', 85)]


## Geographic Coverage Analysis

Analyze scheme availability by state and scheme level. This helps us design citizen queries that test both Central and State government scheme discovery, including location-specific eligibility.

In [11]:
from collections import Counter

central_schemes = [
    s for s in schemes if s.get("level") == "Central"
]

state_schemes = [
    s for s in schemes if s.get("level") == "State"
]

state_distribution = Counter(
    state
    for scheme in state_schemes
    for state in scheme.get("states", [])
    if state != "All"
)

print("Central schemes:", len(central_schemes))
print("State schemes:", len(state_schemes))

print("\nState-wise scheme distribution:")
for state, count in state_distribution.most_common():
    print(f"{state}: {count}")

print("\nCentral scheme state values:")
print(Counter(
    state
    for scheme in central_schemes
    for state in scheme.get("states", [])
))

Central schemes: 693
State schemes: 3735

State-wise scheme distribution:
Gujarat: 592
Uttarakhand: 415
Madhya Pradesh: 261
Goa: 246
Puducherry: 245
Haryana: 224
Tamil Nadu: 206
Rajasthan: 145
West Bengal: 104
Chhattisgarh: 101
Bihar: 101
Jharkhand: 85
Meghalaya: 79
Kerala: 78
Assam: 77
Maharashtra: 77
Odisha: 77
Himachal Pradesh: 73
Dadra & Nagar Haveli and Daman & Diu: 59
Karnataka: 54
Delhi: 49
Andhra Pradesh: 44
Punjab: 41
Uttar Pradesh: 41
Jammu and Kashmir: 39
Tripura: 36
Arunachal Pradesh: 35
Manipur: 24
Sikkim: 21
Mizoram: 21
Telangana: 21
Chandigarh: 19
Andaman and Nicobar Islands: 17
Nagaland: 16
Lakshadweep: 9
Ladakh: 3

Central scheme state values:
Counter({'All': 673, 'Assam': 9, 'Tripura': 7, 'Arunachal Pradesh': 7, 'Manipur': 7, 'Mizoram': 7, 'Tamil Nadu': 6, 'Sikkim': 6, 'Meghalaya': 6, 'Kerala': 6, 'Nagaland': 6, 'Karnataka': 4, 'Jammu and Kashmir': 3, 'Gujarat': 2, 'Rajasthan': 2, 'Madhya Pradesh': 2, 'Chhattisgarh': 2, 'Odisha': 2, 'Maharashtra': 1, 'West Bengal': 1,

## National Geographic Coverage Inventory

Prepare a complete state and union territory coverage inventory from the 4,428-scheme dataset. This will help distribute the citizen-discovery benchmark across India without prioritizing one state.

In [6]:
from collections import Counter

state_scheme_counts = Counter(
    state
    for scheme in schemes
    if scheme.get("level") == "State"
    for state in scheme.get("states", [])
    if state != "All"
)

print("Total states and UTs:", len(state_scheme_counts))
print("\nNational geographic coverage:\n")

for index, (state, count) in enumerate(
    sorted(state_scheme_counts.items()), start=1
):
    print(f"{index:02d}. {state}: {count} schemes")

Total states and UTs: 36

National geographic coverage:

01. Andaman and Nicobar Islands: 17 schemes
02. Andhra Pradesh: 44 schemes
03. Arunachal Pradesh: 35 schemes
04. Assam: 77 schemes
05. Bihar: 101 schemes
06. Chandigarh: 19 schemes
07. Chhattisgarh: 101 schemes
08. Dadra & Nagar Haveli and Daman & Diu: 59 schemes
09. Delhi: 49 schemes
10. Goa: 246 schemes
11. Gujarat: 592 schemes
12. Haryana: 224 schemes
13. Himachal Pradesh: 73 schemes
14. Jammu and Kashmir: 39 schemes
15. Jharkhand: 85 schemes
16. Karnataka: 54 schemes
17. Kerala: 78 schemes
18. Ladakh: 3 schemes
19. Lakshadweep: 9 schemes
20. Madhya Pradesh: 261 schemes
21. Maharashtra: 77 schemes
22. Manipur: 24 schemes
23. Meghalaya: 79 schemes
24. Mizoram: 21 schemes
25. Nagaland: 16 schemes
26. Odisha: 77 schemes
27. Puducherry: 245 schemes
28. Punjab: 41 schemes
29. Rajasthan: 145 schemes
30. Sikkim: 21 schemes
31. Tamil Nadu: 206 schemes
32. Telangana: 21 schemes
33. Tripura: 36 schemes
34. Uttar Pradesh: 41 schemes
35. 

## National Discovery Benchmark Allocation

Create a balanced allocation of 150 citizen queries across all 15 scheme categories and 36 Indian States and Union Territories. Include both Central and State scheme scenarios while ensuring every state and UT is represented.

One important distinction: 10% is our sampling target, not necessarily one unique question per scheme. We should generate questions from scheme information, then verify that each question has the correct relevant scheme IDs. Some citizen questions may legitimately match multiple schemes.

In [12]:
import math
import pandas as pd

state_targets = pd.DataFrame([
    {
        "state": state,
        "scheme_count": count,
        "target_questions": math.ceil(count * 0.10)
    }
    for state, count in state_scheme_counts.items()
])

central_count = len(central_schemes)
central_target = math.ceil(central_count * 0.10)

print(state_targets.to_string(index=False))
print("\nState questions:", state_targets["target_questions"].sum())
print("Central questions:", central_target)
print(
    "Total target questions:",
    state_targets["target_questions"].sum() + central_target
)

NameError: name 'state_scheme_counts' is not defined

## Discovery Benchmark Scheme Sampling

Select approximately 10% of State schemes for each State/UT and 10% of Central schemes. Each selected scheme becomes a source for generating one natural-language citizen query. Preserve the original scheme information and geographic scope for relevance judgments.

In [13]:
import json
import math
import random
from pathlib import Path

random.seed(42)

discovery_dir = Path(
    r"G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery"
)
discovery_dir.mkdir(parents=True, exist_ok=True)

generation_tasks = []

# Sample State schemes separately for each State/UT
for state, count in state_scheme_counts.items():
    state_records = [
        scheme for scheme in schemes
        if scheme.get("level") == "State"
        and state in scheme.get("states", [])
    ]

    target = math.ceil(len(state_records) * 0.10)
    selected = random.sample(state_records, target)

    for scheme in selected:
        generation_tasks.append({
            "generation_id": f"STATE_{state}_{scheme['scheme_id']}",
            "scope": "State",
            "target_state": state,
            "scheme": scheme
        })

# Sample Central schemes
central_target = math.ceil(len(central_schemes) * 0.10)
selected_central = random.sample(
    central_schemes, central_target
)

for scheme in selected_central:
    generation_tasks.append({
        "generation_id": f"CENTRAL_{scheme['scheme_id']}",
        "scope": "Central",
        "target_state": "All India",
        "scheme": scheme
    })

output_path = discovery_dir / "discovery_generation_tasks.jsonl"

with open(output_path, "w", encoding="utf-8") as file:
    for task in generation_tasks:
        file.write(
            json.dumps(task, ensure_ascii=False) + "\n"
        )

print("Generation tasks:", len(generation_tasks))
print("State tasks:", sum(t["scope"] == "State" for t in generation_tasks))
print("Central tasks:", sum(t["scope"] == "Central" for t in generation_tasks))
print("Saved:", output_path)

NameError: name 'state_scheme_counts' is not defined

In [15]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file
import os

# google_api = os.getenv("GOOGLE_API_KEY")
groq_api = os.getenv("GROQ_API_KEY")

if groq_api is None:
    raise ValueError("GROQ_API_KEY is not set in the environment variables")

llm = ChatGroq(
    api_key=groq_api,
    model="openai/gpt-oss-120b"
)

## Scheme Discovery Golden Dataset Schema

Define the standard structure for evaluating citizen-to-scheme discovery.

Each record represents one citizen problem and contains:
- Optional citizen profile filters.
- Natural-language citizen query.
- Multiple relevant scheme IDs.
- Graded relevance judgments for each evaluated scheme.
- Evidence explaining why each scheme is relevant.

This dataset will evaluate scheme discovery, not scheme-specific question answering.

### Pilot Citizen Query Generation

Generate one realistic citizen problem using a sampled government scheme as the
source of inspiration.

The citizen query must describe the problem or need in natural language.
Do not mention the scheme name, scheme ID, or repeat profile filters such as
state, age, caste, or income in the query.

This is only a candidate query. Its relevant schemes will be identified and
verified separately against the complete scheme dataset.

In [15]:
import json

task = generation_tasks[0]
scheme = task["scheme"]

scheme_context = {
    "name": scheme.get("name"),
    "level": scheme.get("level"),
    "states": scheme.get("states"),
    "categories": scheme.get("categories"),
    "target_beneficiaries": scheme.get("target_beneficiaries"),
    "brief_description": scheme.get("brief_description"),
    "benefits": scheme.get("benefits"),
    "eligibility": scheme.get("eligibility"),
}

prompt = f"""
You are creating a golden dataset for an Indian government scheme discovery system.

Generate ONE realistic citizen problem query based on the scheme information below.

SCHEME INFORMATION:
{json.dumps(scheme_context, ensure_ascii=False, indent=2)}

STRICT RULES:
1. Generate one short citizen problem query (8–20 words).
2. Use simple, natural, conversational language.
3. Describe only the citizen's primary need or problem.
4. Do not mention scheme names, IDs, departments, or government scheme descriptions.
5. Do not invent personal details such as age, income, caste, or occupation.
6. Do not repeat profile filters in the query.
7. Avoid unnecessary background, explanations, and multiple questions.
8. Do not ask about eligibility, documents, or application procedures.

Return only valid JSON:
{{
    "citizen_query": "..."
}}
"""

response = llm.invoke(prompt)

result = response.content
if isinstance(result, list):
    result = "".join(
        item.get("text", "") if isinstance(item, dict) else str(item)
        for item in result
    )

result = result.strip().removeprefix("```json").removesuffix("```").strip()
pilot_query = json.loads(result)

print("Source scheme:", scheme.get("name"))
print("Target state:", task.get("target_state"))
print("Generated citizen query:", pilot_query["citizen_query"])

Source scheme: Transport Loan Scheme
Target state: Delhi
Generated citizen query: How can I get a loan to buy a small commercial vehicle?


### Generate Discovery Query Candidates

Generate one short citizen-problem query for each sampled scheme.

Keep the citizen query independent of profile filters. Store the target state
separately as structured profile information.

These queries are candidates only. Relevant scheme IDs must be identified
and validated against the complete scheme dataset before adding them to the
discovery golden dataset.

### Inspect Scheme Categories

Identify the exact category names and scheme distribution from the original
scheme dataset. We will use these categories to organize our few-shot examples.

### Select Social Welfare Few-Shot Examples

Inspect original scheme records in the Social welfare & Empowerment category.
Identify diverse schemes that can demonstrate how original scheme information
is transformed into short, natural citizen problem queries.

In [22]:
import json

task = generation_tasks[0]
scheme = task["scheme"]

scheme_context = {
    "name": scheme.get("name"),
    "level": scheme.get("level"),
    "states": scheme.get("states"),
    "categories": scheme.get("categories"),
    "target_beneficiaries": scheme.get("target_beneficiaries"),
    "brief_description": scheme.get("brief_description"),
    "benefits": scheme.get("benefits"),
    "eligibility": scheme.get("eligibility"),
}

few_shot_examples = [
    {
        "input": {
            "scheme_context": {
                "scheme_id": "3ca0F5gB0AGc0F0iDzOE",
                "name": "Financial Assistance For The Marriage Of Daughters Of Poor Widows And Orphan Girls",
                "level": "State",
                "states": ["Delhi"],
                "categories": [
                    "Social welfare & Empowerment",
                    "Women and Child"
                ],
                "target_beneficiaries": [
                    {"value": "individual", "label": "Individual"}
                ],
                "brief_description": "This is a welfare scheme by the GNCTD to provide financial assistance to poor widows for performing the marriage of their daughters and to provide financial assistance to the guardians including Homes/Institutions or foster parents of an orphan girl or an orphan girl for her marriage.",
                "details": "The scheme provides financial assistance to poor widows for performing the marriage of their daughters (up to two daughters) and to guardians, institutions, foster parents or orphan girls for marriage.",
                "benefits": "One-time financial assistance of ₹30,000, up to two daughters only.",
                "eligibility": "Minimum five years residence in Delhi; annual income below ₹1,00,000; other specified conditions apply."
            }
        },
        "output": {
            "citizen_question": "I need financial help to arrange my daughter's marriage."
        }
    }
]

prompt = f"""
    Generate one realistic citizen problem question.

    Learn the input-output pattern from these examples:

    {json.dumps(few_shot_examples, ensure_ascii=False)}

    INPUT:
    {json.dumps(scheme_context, ensure_ascii=False)}

    Generate a question that represents the citizen's actual need.

    Rules:
    - Keep it short (8–20 words).
    - Use natural language.
    - Do not mention the scheme name.
    - Do not invent personal details.
    - Do not ask about eligibility or application procedures.

    OUTPUT:
    Return only JSON:
    {{"citizen_question": "..."}}
    """

### Test Few-Shot Citizen Query Generation

Use the manually curated marriage-assistance example as a few-shot example.
Pass the complete original Old Age Samman Allowance scheme context to the LLM.

Generate one short citizen problem question.

In [24]:
import json

# Retrieve actual scheme from original dataset
# current_scheme = next(
#     s for s in schemes
#     if s["scheme_id"] == "VaHgF5gBXu-iUKXq63iW"
# )

current_scheme = schemes[5]  # For demonstration, using the first scheme

scheme_context = {
    "scheme_id": current_scheme["scheme_id"],
    "name": current_scheme["name"],
    "level": current_scheme["level"],
    "states": current_scheme["states"],
    "categories": current_scheme["categories"],
    "target_beneficiaries": current_scheme["target_beneficiaries"],
    "brief_description": current_scheme["brief_description"],
    "details": current_scheme["details"],
    "benefits": current_scheme["benefits"],
    "eligibility": current_scheme["eligibility"]
}

prompt = f"""
Generate one realistic citizen problem question.

FEW-SHOT EXAMPLE:
{json.dumps(few_shot_examples[0], ensure_ascii=False)}

INPUT:
{json.dumps({"scheme_context": scheme_context}, ensure_ascii=False)}

Generate a citizen question that follows the example's input-output pattern.

Rules:
- Keep it short (8–20 words).
- Use natural language.
- Focus on the primary need addressed by the scheme.
- Do not mention the scheme name.
- Do not invent personal details.
- Do not ask about eligibility, documents or application procedures.

Return only JSON:
{{"citizen_question": "..."}}
"""

response = llm.invoke(prompt)

result = parse_llm_json(response)

print("Scheme:", current_scheme["name"])
print("Generated citizen question:", result["citizen_question"])

Scheme: Post Matric Scholarship Scheme For The Students Belonging To Scheduled Tribe For Studies In India
Generated citizen question: I need financial support to continue my college studies.


### Build Category-Wise Few-Shot Library FINAL

Select two representative original schemes from each category.
Preserve their original scheme context and prepare the input-output
structure for manually curated citizen questions.

In [17]:
import json
import random

fields = [
    "scheme_id", "name", "level", "states", "categories",
    "target_beneficiaries", "brief_description", "details",
    "benefits", "eligibility"
]

rng = random.Random(42)
selected_ids = set()
fewshot_candidates = {}

for category in category_counts:

    records = [
        s for s in schemes
        if category in (s.get("categories") or [])
        and s["scheme_id"] not in selected_ids
        and s.get("brief_description")
        and s.get("benefits")
    ]

    selected = rng.sample(records, min(2, len(records)))
    fewshot_candidates[category] = []

    for scheme in selected:
        selected_ids.add(scheme["scheme_id"])

        context = {field: scheme.get(field) for field in fields}

        fewshot_candidates[category].append({
            "input": {"scheme_context": context},
            "output": {"citizen_question": ""}
        })

output_path = discovery_dir / "discovery_fewshot_candidates.json"

with open(output_path, "w", encoding="utf-8") as file:
    json.dump(fewshot_candidates, file, ensure_ascii=False, indent=2)

print("Categories:", len(fewshot_candidates))
print("Selected examples:", sum(map(len, fewshot_candidates.values())))
print("Saved:", output_path)

Categories: 15
Selected examples: 30
Saved: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_fewshot_candidates.json


## Step 1: Validate Discovery Few-Shot Dataset

The few-shot dataset provides category-specific examples to guide citizen-question generation. Before using it in the LLM prompt, we validate its structure, category distribution, scheme contexts, and generated questions to catch missing or incomplete examples.

In [18]:
import json
from pathlib import Path
from collections import Counter

# PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")
# DISCOVERY_DIR = PROJECT_ROOT / "notebooks" / "golden_dataset" / "discovery"

FEWSHOT_PATH = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\discovery\\discovery_fewshot_candidates.json"

with open(FEWSHOT_PATH, "r", encoding="utf-8") as f:
    fewshot_data = json.load(f)

total_examples = 0
missing_context = []
missing_questions = []
category_counts = {}

required_context_fields = [
    "scheme_id",
    "name",
    "level",
    "states",
    "categories",
    "target_beneficiaries",
    "brief_description",
    "details",
    "benefits",
    "eligibility"
]

for category, examples in fewshot_data.items():
    category_counts[category] = len(examples)
    total_examples += len(examples)

    for index, example in enumerate(examples):
        context = example.get("input", {}).get("scheme_context", {})
        question = example.get("output", {}).get("citizen_question", "")

        missing_fields = [
            field for field in required_context_fields
            if field not in context or context[field] is None
        ]

        if missing_fields:
            missing_context.append({
                "category": category,
                "example": index + 1,
                "missing_fields": missing_fields
            })

        if not isinstance(question, str) or not question.strip():
            missing_questions.append({
                "category": category,
                "example": index + 1
            })

print("Total categories:", len(fewshot_data))
print("Total examples:", total_examples)
print("Category distribution:", category_counts)
print("Missing context:", len(missing_context))
print("Missing questions:", len(missing_questions))

if missing_context:
    print("\nIncomplete contexts:")
    print(missing_context)

if missing_questions:
    print("\nMissing citizen questions:")
    print(missing_questions)

assert len(fewshot_data) == 15, "Expected 15 categories"
assert total_examples == 30, "Expected 30 examples"
assert not missing_context, "Some scheme contexts are incomplete"
assert not missing_questions, "Some citizen questions are missing"

print("\nDataset structural validation passed.")

Total categories: 15
Total examples: 30
Category distribution: {'Business & Entrepreneurship': 2, 'Banking,Financial Services and Insurance': 2, 'Social welfare & Empowerment': 2, 'Sports & Culture': 2, 'Women and Child': 2, 'Education & Learning': 2, 'Agriculture,Rural & Environment': 2, 'Skills & Employment': 2, 'Health & Wellness': 2, 'Housing & Shelter': 2, 'Public Safety,Law & Justice': 2, 'Science, IT & Communications': 2, 'Transport & Infrastructure': 2, 'Travel & Tourism': 2, 'Utility & Sanitation': 2}
Missing context: 0
Missing questions: 30

Missing citizen questions:
[{'category': 'Business & Entrepreneurship', 'example': 1}, {'category': 'Business & Entrepreneurship', 'example': 2}, {'category': 'Banking,Financial Services and Insurance', 'example': 1}, {'category': 'Banking,Financial Services and Insurance', 'example': 2}, {'category': 'Social welfare & Empowerment', 'example': 1}, {'category': 'Social welfare & Empowerment', 'example': 2}, {'category': 'Sports & Culture',

AssertionError: Some citizen questions are missing

## Category-Based Few-Shot Selection

Select few-shot examples by matching the target scheme's categories against the available few-shot dataset. No embeddings or additional classification model are required.

In [35]:
def select_fewshot_examples(scheme, fewshot_data, max_examples=4):
    scheme_categories = scheme.get("categories", [])

    selected = []

    for category in scheme_categories:
        examples = fewshot_data.get(category, [])

        for example in examples:
            selected.append({
                "category": category,
                "scheme_context": example["input"]["scheme_context"],
                "citizen_question": example["output"]["citizen_question"]
            })

    return selected[:max_examples]


# Test
selected_examples = select_fewshot_examples(
    test_scheme,
    fewshot_data,
    max_examples=4
)

for example in selected_examples:
    print(example["category"])
    print(example["scheme_context"]["name"])
    print(example["citizen_question"])
    print("-" * 50)

Business & Entrepreneurship
25% Capital Investment Subsidy Scheme
I want to start a small business but need financial help to set it up.
--------------------------------------------------
Business & Entrepreneurship
Scheme for Development Maintenance and Construction of Roads
My small business handles road construction, and I need a loan to purchase equipment.
--------------------------------------------------
Banking,Financial Services and Insurance
Green Business Scheme
I want to start an eco-friendly business but need an affordable loan to fund it.
--------------------------------------------------
Banking,Financial Services and Insurance
Term Loan Scheme (GTKVN) 
I need a loan to start a small business or support my traditional occupation.
--------------------------------------------------


In [ ]:
import json

current_scheme = schemes[5]  # For demonstration, using the first scheme

scheme_context = {
    "scheme_id": current_scheme["scheme_id"],
    "name": current_scheme["name"],
    "level": current_scheme["level"],
    "states": current_scheme["states"],
    "categories": current_scheme["categories"],
    "target_beneficiaries": current_scheme["target_beneficiaries"],
    "brief_description": current_scheme["brief_description"],
    "details": current_scheme["details"],
    "benefits": current_scheme["benefits"],
    "eligibility": current_scheme["eligibility"]
}

prompt = f"""
Generate one realistic citizen problem question.

FEW-SHOT EXAMPLE:
{json.dumps(few_shot_examples[0], ensure_ascii=False)}

INPUT:
{json.dumps({"scheme_context": scheme_context}, ensure_ascii=False)}

Generate a citizen question that follows the example's input-output pattern.

Rules:
- Keep it short (8–20 words).
- Use natural language.
- Focus on the primary need addressed by the scheme.
- Do not mention the scheme name.
- Do not invent personal details.
- Do not ask about eligibility, documents or application procedures.

Return only JSON:
{{"citizen_question": "..."}}
"""

response = llm.invoke(prompt)

result = parse_llm_json(response)

print("Scheme:", current_scheme["name"])
print("Generated citizen question:", result["citizen_question"])

In [19]:
import json
import time
from pathlib import Path

from tqdm.auto import tqdm
from langchain_core.messages import HumanMessage

PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

SCHEMES_PATH = (
    PROJECT_ROOT / "notebooks/data/analysis/all_schemes.json"
)

FEWSHOT_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "discovery_fewshot_candidates.json"
)

OUTPUT_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "generated_discovery_questions.jsonl"
)

MAX_EXAMPLES = 4
QUESTIONS_PER_SCHEME = 5

GOLDEN_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "discovery_golden_dataset.jsonl"
)

with open(GOLDEN_PATH, "r", encoding="utf-8") as f:
    golden_schemes = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Golden dataset schemes:", len(golden_schemes))

with open(FEWSHOT_PATH, "r", encoding="utf-8") as f:
    fewshot_data = json.load(f)

print("Total schemes:", len(schemes))
print("Few-shot categories:", len(fewshot_data))


def select_fewshot_examples(
    scheme,
    fewshot_data,
    max_examples=MAX_EXAMPLES
):
    scheme_categories = scheme.get("categories", [])

    selected = []

    for category in scheme_categories:
        examples = fewshot_data.get(category, [])

        for example in examples:
            selected.append({
                "category": category,
                "scheme_context": example["input"]["scheme_context"],
                "citizen_question": example["output"]["citizen_question"]
            })

    return selected[:max_examples]



def build_discovery_prompt(target_scheme, selected_examples):

    examples_text = "\n\n".join(
        [
            f"Scheme Context:\n"
            f"{json.dumps(example['scheme_context'], ensure_ascii=False)}\n"
            f"Citizen Question:\n"
            f"{example['citizen_question']}"
            for example in selected_examples
        ]
    )

    target_context = json.dumps(
        target_scheme,
        ensure_ascii=False
    )

    return f"""
You are an expert in understanding Indian government welfare schemes.

Your task is to generate realistic citizen queries based on the target scheme.

FEW-SHOT EXAMPLES:
{examples_text}

TARGET SCHEME:
{target_context}

INSTRUCTIONS:
1. Generate 5 different natural citizen queries.
2. Each query should describe a real citizen need addressed by the scheme.
3. Keep each query between 8 and 20 words.
4. Do not mention the scheme name.
5. Do not invent personal details, demographics, or circumstances.
6. Do not ask about eligibility, documents, or application procedures.
7. Avoid repeating state, caste, age, and income filters because these are collected separately.
8. Follow the style of the examples without copying their questions.
9. Return valid JSON only.

OUTPUT FORMAT:
{{
    "citizen_questions": [
        "Question 1",
        "Question 2",
        "Question 3",
        "Question 4",
        "Question 5"
    ]
}}
"""

Golden dataset schemes: 0
Total schemes: 4428
Few-shot categories: 15


In [23]:
import json
PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

SCHEMES_PATH = "G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\discovery\\discovery_generation_tasks.jsonl"

with open(SCHEMES_PATH, "r", encoding="utf-8") as f:
    g_scheme_data = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

DISCOVERY_FIELDS = [
    "scheme_id",
    "name",
    "level",
    "states",
    "categories",
    "tags",
    "target_beneficiaries",
    "brief_description",
    "details",
    "benefits"
]

discovery_schemes = [
    {
        field: record["scheme"].get(field)
        for field in DISCOVERY_FIELDS
    }
    for record in g_scheme_data
]

In [24]:
discovery_schemes[10]

{'scheme_id': 'Bhe7F5gB5uF80_MGVZFX',
 'name': 'Financial Assistance for Artificial Limbs to Workers and their Dependents- Haryana Labour Welfare Board',
 'level': 'State',
 'states': ['Haryana'],
 'categories': ['Social welfare & Empowerment'],
 'tags': ['Worker',
  'Financial Assistance',
  'Artificial Limbs',
  'PwD',
  'Disability',
  'Labour'],
 'target_beneficiaries': [{'value': 'individual', 'label': 'Individual'},
  {'value': 'family', 'label': 'Family'}],
 'brief_description': 'Under this scheme, financial assistance is provided for artificial limbs upto rates fixed by Saket Hospital, Chandimandir(Panchkula). This assistance is available to workers employed in industrial &commercial units in Haryana and their dependents, who become disabled due to the loss of vital organs.',
 'details': 'The scheme “Financial Assistance for Artificial Limbs to Workers and their Dependents” is implemented by the Haryana Labour Welfare Board (HLWB), Labour Department, Government of Haryana. Unde

In [25]:
def generate_citizen_questions(
    target_scheme,
    selected_examples
):

    prompt = build_discovery_prompt(
        target_scheme,
        selected_examples
    )

    response = llm.invoke(prompt)

    content = response.content

    if isinstance(content, list):
        content = "".join(
            item.get("text", "")
            for item in content
            if isinstance(item, dict)
        )

    content = content.strip()

    # Handle accidental Markdown JSON fences
    if content.startswith("```"):
        content = content.split("\n", 1)[1]
        content = content.rsplit("```", 1)[0]

    result = json.loads(content)

    questions = result["citizen_questions"]

    if len(questions) != QUESTIONS_PER_SCHEME:
        raise ValueError(
            f"Expected 5 questions, received {len(questions)}"
        )

    return questions

### Generate questions for all schemes
Generate five citizen questions for each selected scheme using category-based few-shot examples. Save each successful result immediately so generation can resume if interrupted.

In [29]:

import json
import time
from pathlib import Path
from tqdm.auto import tqdm

BATCH_SIZE = 10
MAX_RETRIES = 3
RETRY_DELAY = 30

PROJECT_ROOT = Path(r"G:\SEVA_Ai\SEVA_Ai")

OUTPUT_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "generated_discovery_questions.jsonl"
)

CHECKPOINT_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "generation_checkpoint.json"
)

ERROR_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "generation_errors.jsonl"
)


def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            return json.load(f)

    return {"completed_scheme_ids": []}


def save_checkpoint(completed_ids):
    temp_path = CHECKPOINT_PATH.with_suffix(".tmp")

    with open(temp_path, "w", encoding="utf-8") as f:
        json.dump(
            {"completed_scheme_ids": list(completed_ids)},
            f,
            ensure_ascii=False,
            indent=2
        )

    temp_path.replace(CHECKPOINT_PATH)


def generate_one_scheme(scheme):

    examples = select_fewshot_examples(
        scheme,
        fewshot_data,
        max_examples=4
    )

    prompt = build_discovery_prompt(scheme, examples)

    for attempt in range(MAX_RETRIES):

        try:
            response = llm.invoke(prompt)
            content = response.content.strip()

            # Handle Markdown JSON fences
            if content.startswith("```"):
                content = content.split("\n", 1)[1]
                content = content.rsplit("```", 1)[0]

            result = json.loads(content)
            questions = result["citizen_questions"]

            if len(questions) != 5:
                raise ValueError(
                    f"Expected 5 questions, got {len(questions)}"
                )

            return {
                "scheme_id": scheme["scheme_id"],
                "scheme_name": scheme["name"],
                "categories": scheme["categories"],
                "citizen_questions": questions
            }

        except Exception as e:

            if attempt == MAX_RETRIES - 1:
                raise e

            print(
                f"Retry {attempt + 1} for "
                f"{scheme['scheme_id']}: {e}"
            )

            time.sleep(RETRY_DELAY * (attempt + 1))


def generate_in_batches(schemes):

    checkpoint = load_checkpoint()

    completed_ids = set(
        checkpoint["completed_scheme_ids"]
    )

    # Reconcile checkpoint against output file
    if OUTPUT_PATH.exists():
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    completed_ids.add(record["scheme_id"])

    save_checkpoint(completed_ids)

    remaining = [
        scheme for scheme in schemes
        if scheme["scheme_id"] not in completed_ids
    ]

    print("Total schemes:", len(schemes))
    print("Already completed:", len(completed_ids))
    print("Remaining:", len(remaining))

    for batch_start in range(0, len(remaining), BATCH_SIZE):

        batch = remaining[
            batch_start:batch_start + BATCH_SIZE
        ]

        print(
            f"\nBatch {batch_start // BATCH_SIZE + 1} "
            f"| Schemes: {len(batch)}"
        )

        for scheme in tqdm(batch):

            scheme_id = scheme["scheme_id"]

            try:
                record = generate_one_scheme(scheme)

                # Save result immediately
                with open(
                    OUTPUT_PATH, "a", encoding="utf-8"
                ) as f:
                    f.write(
                        json.dumps(record, ensure_ascii=False) + "\n"
                    )

                # Update checkpoint only after successful save
                completed_ids.add(scheme_id)
                save_checkpoint(completed_ids)

            except Exception as e:

                error_record = {
                    "scheme_id": scheme_id,
                    "error": str(e)
                }

                with open(
                    ERROR_PATH, "a", encoding="utf-8"
                ) as f:
                    f.write(
                        json.dumps(error_record, ensure_ascii=False)
                        + "\n"
                    )

                print(f"\nFailed: {scheme_id} | {e}")

                # Stop the run if the API quota/rate limit is exhausted
                error_text = str(e).lower()

                if any(term in error_text for term in [
                    "rate_limit",
                    "rate limit",
                    "resource_exhausted",
                    "quota",
                    "429",
                    "too many requests"
                ]):
                    print("API limit detected. Stopping safely.")
                    return

        print(
            "Completed:",
            len(completed_ids),
            "/",
            len(schemes)
        )

    print("\nGeneration run finished.")
    print("Completed schemes:", len(completed_ids))
    print("Remaining schemes:", len(schemes) - len(completed_ids))


In [30]:
generate_in_batches(discovery_schemes)

Total schemes: 462
Already completed: 79
Remaining: 383

Batch 1 | Schemes: 10


  0%|          | 0/10 [00:00<?, ?it/s]

Completed: 89 / 462

Batch 2 | Schemes: 10


  0%|          | 0/10 [00:00<?, ?it/s]

Retry 1 for Ghc9GJgB5uF80_MGB5Rw: Connection error.
Completed: 99 / 462

Batch 3 | Schemes: 10


  0%|          | 0/10 [00:00<?, ?it/s]

Completed: 109 / 462

Batch 4 | Schemes: 10


  0%|          | 0/10 [00:00<?, ?it/s]

Retry 1 for uca9F5gB0AGc0F0i3zVe: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jvs1s1wvf6dascjvb1hnywth` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 5711, Requested 2351. Please try again in 464.999999ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Completed: 119 / 462

Batch 5 | Schemes: 10


  0%|          | 0/10 [00:00<?, ?it/s]

Retry 1 for bBfpF5gB5uF80_MGYpPK: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jvs1s1wvf6dascjvb1hnywth` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198771, Requested 2172. Please try again in 6m47.376s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}
Retry 2 for bBfpF5gB5uF80_MGYpPK: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jvs1s1wvf6dascjvb1hnywth` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 198701, Requested 2172. Please try again in 6m17.136s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

Failed: bBfpF5gB5uF80_MGYpPK | Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-12

In [31]:
import json
from pathlib import Path
from collections import Counter

DATA_PATH = (
    PROJECT_ROOT / "notebooks/golden_dataset/discovery/"
    "generated_discovery_questions_final.jsonl"
)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    generated_data = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

scheme_ids = [
    record["scheme_id"]
    for record in generated_data
]

questions = [
    question.strip()
    for record in generated_data
    for question in record["citizen_questions"]
]

question_counts = Counter(questions)
scheme_counts = Counter(scheme_ids)

duplicate_questions = {
    q: count
    for q, count in question_counts.items()
    if count > 1
}

word_lengths = [len(q.split()) for q in questions]

print("Total scheme records:", len(generated_data))
print("Unique scheme IDs:", len(set(scheme_ids)))
print("Total questions:", len(questions))
print("Unique question texts:", len(question_counts))
print("Duplicate question texts:", len(duplicate_questions))
print("Duplicate scheme IDs:", sum(c > 1 for c in scheme_counts.values()))
print("Questions per scheme:", Counter(
    len(r["citizen_questions"]) for r in generated_data
))
print("Minimum words:", min(word_lengths))
print("Maximum words:", max(word_lengths))

print("\nTop 20 repeated questions:")
for question, count in question_counts.most_common(20):
    if count > 1:
        print(f"{count} times: {question}")

Total scheme records: 462
Unique scheme IDs: 462
Total questions: 2310
Unique question texts: 944
Duplicate question texts: 141
Duplicate scheme IDs: 0
Questions per scheme: Counter({5: 462})
Minimum words: 8
Maximum words: 16

Top 20 repeated questions:
49 times: I need support to prepare for competitive examinations with coaching support.
49 times: I am looking for help to prepare for competitive examinations with coaching support.
49 times: How can I get assistance to prepare for competitive examinations with coaching support?
49 times: I want to prepare for competitive examinations with coaching support and need help.
49 times: Where can I find support to prepare for competitive examinations with coaching support?
48 times: I need support to continue my education and manage study expenses.
48 times: I am looking for help to continue my education and manage study expenses.
48 times: How can I get assistance to continue my education and manage study expenses?
48 times: I want to cont

In [32]:
from collections import defaultdict

question_schemes = defaultdict(list)

for record in generated_data:
    for question in record["citizen_questions"]:
        question_schemes[question].append({
            "scheme_id": record["scheme_id"],
            "scheme_name": record["scheme_name"]
        })

duplicates = {
    question: schemes
    for question, schemes in question_schemes.items()
    if len(schemes) > 1
}

for question, schemes in sorted(
    duplicates.items(),
    key=lambda x: len(x[1]),
    reverse=True
)[:10]:

    print(f"\nQuestion: {question}")
    print(f"Used by {len(schemes)} schemes")

    for scheme in schemes:
        print(" -", scheme["scheme_name"])


Question: I need support to prepare for competitive examinations with coaching support.
Used by 49 schemes
 - Grant on Passing the Written Examination of Competitive Examinations for Recruitment in Armed Forces
 - Mukhyamantri Shiksha Alankar Yojana for Students of Economically Weaker Section
 - Swami Vivekananda Merit-cum-Means Scholarship
 - Akanksha Housing Scheme
 - Mahatma Jyotirao Phule Jan Arogya Yojana
 - Lakshya Scheme
 - Mukhyamantri Bagwani Mission: Dragon Fruit Vikas Yojana
 - Old Age Home (Sahara)
 - Free Uniforms for Students
 - Capital Investment Subsidy for Establishment of Way Side Amenities Under Green Field or Franchise Model
 - Tool/Equipment Purchase Subsidy Scheme (MPBOCWWB)
 - Shauchalaya Sahayata Yojana
 - Scheme for Goa Rajya Vidnyanik Puraskar
 - Bhagwan Birsa Munda Lakshya Siddhi Yojana
 - Promotion of Patriotism/National Integration/Communal Harmony - Production of Musical Programme or Tiatr
 - Postmatric Scholarship
 - Grant of Quality Certification Reimbu

## Discovery Golden Dataset — Question-to-Scheme Validation

Purpose:
- Load the 131 Groq-generated scheme records.
- Match each scheme with its original source information.
- Attach scheme descriptions, benefits, eligibility and other relevant context.
- Prepare individual question records for relevance validation.

The original scheme information will be the source of truth.
No questions or relevance labels will be modified in this step.

In [37]:

import json
from pathlib import Path

BASE_DIR = Path(r"G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery")

GENERATED_PATH = BASE_DIR / "generated_discovery_questions.jsonl"
TASKS_PATH = BASE_DIR / "discovery_generation_tasks.jsonl"
OUTPUT_PATH = BASE_DIR / "discovery_131_validation_input.jsonl"

def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

generated_data = load_jsonl(GENERATED_PATH)
generation_tasks = load_jsonl(TASKS_PATH)

# Index original scheme information by scheme ID
scheme_lookup = {
    task["scheme"]["scheme_id"]: task["scheme"]
    for task in generation_tasks
}

validation_records = []

for record in generated_data:
    scheme_id = record["scheme_id"]
    scheme = scheme_lookup.get(scheme_id)

    if scheme is None:
        print(f"Missing scheme: {scheme_id}")
        continue

    scheme_context = {
        field: scheme.get(field)
        for field in [
            "scheme_id",
            "name",
            "level",
            "states",
            "categories",
            "tags",
            "target_beneficiaries",
            "brief_description",
            "details",
            "benefits"
        ]
    }

    for question_id, question in enumerate(
        record["citizen_questions"], start=1
    ):
        validation_records.append({
            "scheme_id": scheme_id,
            "scheme_name": record["scheme_name"],
            "question_id": question_id,
            "question": question,
            "scheme_context": scheme_context
        })

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for item in validation_records:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("Generated scheme records:", len(generated_data))
print("Matched schemes:", len(set(r["scheme_id"] for r in validation_records)))
print("Validation questions:", len(validation_records))
print("Output:", OUTPUT_PATH)


Generated scheme records: 121
Matched schemes: 121
Validation questions: 605
Output: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\discovery_131_validation_input.jsonl


In [39]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

providers = {
    "Groq": "GROQ_API_KEY",
    "Gemini": "GEMINI_API_KEY",
    "OpenAI": "OPENAI_API_KEY",
}

for provider, key in providers.items():
    status = "Available" if os.getenv(key) else "Missing"
    print(f"{provider}: {status}")

Groq: Available
Gemini: Available
OpenAI: Available


In [40]:

import os
from dotenv import load_dotenv
from litellm import Router

load_dotenv(override=True)

model_list = [
    {
        "model_name": "seva-judge",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",
            "api_key": os.environ["GROQ_API_KEY"],
        },
    },
    {
        "model_name": "seva-judge-gemini",
        "litellm_params": {
            "model": "gemini/gemini-3.8-flash",
            "api_key": os.environ["GEMINI_API_KEY"],
        },
    },
    {
        "model_name": "seva-judge-openai",
        "litellm_params": {
            "model": "openai/gpt-6.2-sol",
            "api_key": os.environ["OPENAI_API_KEY"],
        },
    },
]

router = Router(
    model_list=model_list,
    fallbacks=[
        {
            "seva-judge": [
                "seva-judge-gemini",
                "seva-judge-openai",
            ]
        }
    ],
    num_retries=1,
    timeout=60,
)

print("LiteLLM Router configured successfully!")


LiteLLM Router configured successfully!


In [ ]:

import json
from typing import Literal

from pydantic import BaseModel, Field


class SchemeQuestionJudge(BaseModel):
    relevance: Literal[
        "relevant",
        "partially_relevant",
        "not_relevant"
    ]
    reason: str
    unsupported_claims: list[str] = Field(default_factory=list)
    confidence: Literal["high", "medium", "low"]


def parse_judge_response(content):
    if not content or not content.strip():
        raise ValueError("Empty model response")

    content = content.strip()

    if content.startswith("```"):
        content = content.split("\n", 1)[1]
        content = content.rsplit("```", 1)[0].strip()

    start = content.find("{")

    if start == -1:
        raise ValueError("No JSON object found")

    payload, _ = json.JSONDecoder().raw_decode(content[start:])

    return SchemeQuestionJudge.model_validate(payload)


def validate_scheme_question(record):
    scheme_context = json.dumps(
        record["scheme_context"],
        ensure_ascii=False,
        indent=2
    )

    prompt = f"""
You are a strict evidence-grounded evaluator for an Indian
government schemes discovery platform.

TASK:
Evaluate whether the citizen question is relevant to its assigned scheme.

SCHEME INFORMATION:
{scheme_context}

CITIZEN QUESTION:
{record["question"]}

CLASSIFICATION RULES:

1. relevant:
The question clearly relates to the scheme's documented purpose,
benefits, eligibility, beneficiaries, or application process.
Its assumptions are supported by the source.

2. partially_relevant:
The question is related to the scheme but contains specific
assumptions or claims not supported by the source.

3. not_relevant:
The question is unrelated to the scheme or contradicts its
documented purpose and scope.

IMPORTANT:
- Use ONLY explicitly documented information in SCHEME INFORMATION.
- Never use external knowledge to classify vehicle types,
  occupations, beneficiary groups, or geographical eligibility.
- Never assume that two similar terms are equivalent.
- A vehicle must be explicitly supported by the scheme source.
- If the scheme mentions Light Motor Vehicles but does not
  specify whether motorbikes, auto-rickshaws, or vans qualify,
  do not assume they qualify.
- Verify geographical applicability against the scheme's
  states field whenever a question specifies a location.
- Distinguish between a question's general topic and its
  specific factual assumptions.
- If the main topic matches but a specific detail is unsupported,
  classify it as partially_relevant.
- If a central requirement contradicts the source, classify
  it as not_relevant.
- Do not invent evidence to justify relevance.
- Every relevance decision must be supported by source evidence.

Return ONLY valid JSON in this format:

{{
    "relevance": "relevant",
    "reason": "Brief evidence-based explanation",
    "unsupported_claims": [],
    "confidence": "high"
}}
"""

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    model_aliases = [
        "seva-judge",
        "seva-judge-gemini",
        "seva-judge-openai"
    ]

    errors = []

    for model_alias in model_aliases:
        try:
            response = router.completion(
                model=model_alias,
                messages=messages,
                temperature=0,
                max_tokens=1200
            )

            content = response.choices[0].message.content

            result = parse_judge_response(content)

            return {
                **record,
                "validation": result.model_dump(),
                "judge_model": response.model,
                "validation_status": "completed"
            }

        except Exception as exc:
            errors.append(
                f"{model_alias}: {type(exc).__name__}: {str(exc)[:250]}"
            )

    raise RuntimeError(
        "All validation attempts failed:\n" + "\n".join(errors)
    )


# Load the validation input
INPUT_PATH = (
    BASE_DIR / "discovery_131_validation_input.jsonl"
)

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    validation_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("Total questions:", len(validation_records))

# Test only the first question
test_result = validate_scheme_question(validation_records[0])

print("\nQuestion:", test_result["question"])
print("Scheme:", test_result["scheme_name"])
print("Judge:", test_result["judge_model"])
print("\nValidation:")
print(json.dumps(
    test_result["validation"],
    indent=2,
    ensure_ascii=False
))


Total questions: 605

Question: I need a loan to purchase a light motor vehicle for my transport business.
Scheme: Transport Loan Scheme
Judge: openai/gpt-oss-120b

Validation:
{
  "relevance": "relevant",
  "reason": "The scheme explicitly offers term loans up to ₹5,00,000 for the purchase of Light Motor Vehicles, matching the citizen's request for a loan to buy an LMV for a transport business.",
  "unsupported_claims": [],
  "confidence": "high"
}


In [49]:
test_batch = validation_records[:10]

batch_results = []

for i, record in enumerate(test_batch, start=1):
    try:
        result = validate_scheme_question(record)
        batch_results.append(result)

        print(
            f"{i}/10 | "
            f"{result['validation']['relevance']} | "
            f"{result['judge_model']}"
        )

    except Exception as e:
        print(f"{i}/10 | ERROR: {e}")

print("\nSuccessful:", len(batch_results))

13:07:31 - LiteLLM Router:ERROR: walk.py:362 - litellm.router.py::async_function_with_fallbacks() - Error occurred while trying to do fallbacks - litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_reasoning_effort. To drop unsupported params set `litellm.drop_params = True`
Debug Information:
Cooldown Deployments=[]
13:07:31 - LiteLLM Router:ERROR: walk.py:362 - litellm.router.py::async_function_with_fallbacks() - Error occurred while trying to do fallbacks - litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_reasoning_effort. To drop unsupported params set `litellm.drop_params = True`
Debug Infor

1/10 | ERROR: All validation attempts failed:
seva-judge: RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jvs1s1wvf6dascjvb1hnywth` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 1990
seva-judge-gemini: UnsupportedParamsError: litellm.UnsupportedParamsError: gemini does not support parameters: ['temperature'], for model=gemini-3.8-flash. To drop these, set `litellm.drop_params=True` or for proxy:

`litellm_settings:
 drop_params: true`
. 
 If you want to use these params d
seva-judge-openai: UnsupportedParamsError: litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_r
2/10 | relevant | openai/gpt-oss-120b


13:07:35 - LiteLLM Router:ERROR: walk.py:362 - litellm.router.py::async_function_with_fallbacks() - Error occurred while trying to do fallbacks - litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_reasoning_effort. To drop unsupported params set `litellm.drop_params = True`
Debug Information:
Cooldown Deployments=[]
13:07:35 - LiteLLM Router:ERROR: walk.py:362 - litellm.router.py::async_function_with_fallbacks() - Error occurred while trying to do fallbacks - litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_reasoning_effort. To drop unsupported params set `litellm.drop_params = True`
Debug Infor

3/10 | ERROR: All validation attempts failed:
seva-judge: RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jvs1s1wvf6dascjvb1hnywth` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 1999
seva-judge-gemini: UnsupportedParamsError: litellm.UnsupportedParamsError: gemini does not support parameters: ['temperature'], for model=gemini-3.8-flash. To drop these, set `litellm.drop_params=True` or for proxy:

`litellm_settings:
 drop_params: true`
. 
 If you want to use these params d
seva-judge-openai: UnsupportedParamsError: litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_r


13:07:38 - LiteLLM Router:ERROR: walk.py:362 - litellm.router.py::async_function_with_fallbacks() - Error occurred while trying to do fallbacks - litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_reasoning_effort. To drop unsupported params set `litellm.drop_params = True`
Debug Information:
Cooldown Deployments=[]
13:07:38 - LiteLLM Router:ERROR: walk.py:362 - litellm.router.py::async_function_with_fallbacks() - Error occurred while trying to do fallbacks - litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_reasoning_effort. To drop unsupported params set `litellm.drop_params = True`
Debug Infor

4/10 | ERROR: All validation attempts failed:
seva-judge: RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jvs1s1wvf6dascjvb1hnywth` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 1999
seva-judge-gemini: UnsupportedParamsError: litellm.UnsupportedParamsError: gemini does not support parameters: ['temperature'], for model=gemini-3.8-flash. To drop these, set `litellm.drop_params=True` or for proxy:

`litellm_settings:
 drop_params: true`
. 
 If you want to use these params d
seva-judge-openai: UnsupportedParamsError: litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_r


... (8 outputs left out)

13:07:50 - LiteLLM Router:ERROR: walk.py:362 - litellm.router.py::async_function_with_fallbacks() - Error occurred while trying to do fallbacks - litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_reasoning_effort. To drop unsupported params set `litellm.drop_params = True`
Debug Information:
Cooldown Deployments=[]
13:07:50 - LiteLLM Router:ERROR: walk.py:362 - litellm.router.py::async_function_with_fallbacks() - Error occurred while trying to do fallbacks - litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_reasoning_effort. To drop unsupported params set `litellm.drop_params = True`
Debug Infor

9/10 | ERROR: All validation attempts failed:
seva-judge: RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jvs1s1wvf6dascjvb1hnywth` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 1999
seva-judge-gemini: UnsupportedParamsError: litellm.UnsupportedParamsError: gemini does not support parameters: ['temperature'], for model=gemini-3.8-flash. To drop these, set `litellm.drop_params=True` or for proxy:

`litellm_settings:
 drop_params: true`
. 
 If you want to use these params d
seva-judge-openai: UnsupportedParamsError: litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_r


13:07:53 - LiteLLM Router:ERROR: walk.py:362 - litellm.router.py::async_function_with_fallbacks() - Error occurred while trying to do fallbacks - litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_reasoning_effort. To drop unsupported params set `litellm.drop_params = True`
Debug Information:
Cooldown Deployments=[]
13:07:53 - LiteLLM Router:ERROR: walk.py:362 - litellm.router.py::async_function_with_fallbacks() - Error occurred while trying to do fallbacks - litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_reasoning_effort. To drop unsupported params set `litellm.drop_params = True`
Debug Infor

10/10 | ERROR: All validation attempts failed:
seva-judge: RateLimitError: litellm.RateLimitError: RateLimitError: GroqException - {"error":{"message":"Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01jvs1s1wvf6dascjvb1hnywth` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 1999
seva-judge-gemini: UnsupportedParamsError: litellm.UnsupportedParamsError: gemini does not support parameters: ['temperature'], for model=gemini-3.8-flash. To drop these, set `litellm.drop_params=True` or for proxy:

`litellm_settings:
 drop_params: true`
. 
 If you want to use these params d
seva-judge-openai: UnsupportedParamsError: litellm.UnsupportedParamsError: gpt-6.2-sol doesn't support temperature=0 while reasoning is active. Only temperature=1 is supported unless reasoning_effort resolves to 'none', either set explicitly on the request or declared as the model's default_r

Successful: 1


In [50]:
from collections import Counter

test_summary = Counter(
    r["validation"]["relevance"]
    for r in batch_results
)

print("Relevance distribution:")
for label in [
    "relevant",
    "partially_relevant",
    "not_relevant"
]:
    print(f"{label}: {test_summary[label]}")

print("\nQuestion-level results:")
for r in batch_results:
    print("-" * 60)
    print("Scheme:", r["scheme_name"])
    print("Question:", r["question"])
    print("Classification:", r["validation"]["relevance"])
    print("Reason:", r["validation"]["reason"])
    print("Unsupported:", r["validation"]["unsupported_claims"])

Relevance distribution:
relevant: 1
partially_relevant: 0
not_relevant: 0

Question-level results:
------------------------------------------------------------
Scheme: Transport Loan Scheme
Question: Looking for financing to buy a small van for goods delivery in Delhi.
Classification: relevant
Reason: The scheme provides term loans up to ₹5,00,000 for purchase of Light Motor Vehicles in Delhi, which directly matches the citizen's request for financing a small van for goods delivery.
Unsupported: []


In [52]:

import json
from pathlib import Path
from collections import Counter

BASE_DIR = Path(
    r"G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery"
)

INPUT_PATH = BASE_DIR / "generated_discovery_questions.jsonl"
OUTPUT_PATH = BASE_DIR / "seva_discovery_golden_v1.jsonl"


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


generated_data = load_jsonl(INPUT_PATH)

golden_records = []
test_counter = 1

for scheme_record in generated_data:

    scheme_id = scheme_record["scheme_id"]
    scheme_name = scheme_record["scheme_name"]
    categories = scheme_record.get("categories", [])

    for question in scheme_record["citizen_questions"]:

        golden_records.append({
            "test_id": f"DISC-{test_counter:04d}",
            "scheme_id": scheme_id,
            "scheme_name": scheme_name,
            "question": question,
            "relevant_scheme_ids": [scheme_id],
            "categories": categories,
            "source": "groq_generated",
            "label_status": "provisional_llm_generated"
        })

        test_counter += 1


# Save JSONL
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    for record in golden_records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


# Dataset validation
scheme_counts = Counter(r["scheme_id"] for r in golden_records)
test_ids = [r["test_id"] for r in golden_records]

assert len(generated_data) == 121
assert len(golden_records) == 605
assert len(scheme_counts) == 121
assert all(count == 5 for count in scheme_counts.values())
assert len(test_ids) == len(set(test_ids))

print("Golden dataset created successfully!")
print("Total schemes:", len(scheme_counts))
print("Total questions:", len(golden_records))
print("Questions per scheme:", set(scheme_counts.values()))
print("Unique test IDs:", len(set(test_ids)))
print("Output:", OUTPUT_PATH)

print("\nSample record:")
print(json.dumps(golden_records[0], indent=2, ensure_ascii=False))


Golden dataset created successfully!
Total schemes: 121
Total questions: 605
Questions per scheme: {5}
Unique test IDs: 605
Output: G:\SEVA_Ai\SEVA_Ai\notebooks\golden_dataset\discovery\seva_discovery_golden_v1.jsonl

Sample record:
{
  "test_id": "DISC-0001",
  "scheme_id": "3he_F5gB5uF80_MGg5Hz",
  "scheme_name": "Transport Loan Scheme",
  "question": "I need a loan to purchase a light motor vehicle for my transport business.",
  "relevant_scheme_ids": [
    "3he_F5gB5uF80_MGg5Hz"
  ],
  "categories": [
    "Transport & Infrastructure",
    "Business & Entrepreneurship"
  ],
  "source": "groq_generated",
  "label_status": "provisional_llm_generated"
}
